# Ordered Logistic Regression Results for Adoption Predictors Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the "Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya" dataset using the `mlcroissant` library. We will step through loading metadata, examining record sets, extracting data, and conducting exploratory data analysis (EDA). 

### Dataset Source
The FAIR<sup>2</sup> dataset package is described via a [Croissant schema JSON-LD file](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json).

In [ ]:
# Install mlcroissant. Remove '--user' if not needed, depending on your environment.
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and inspect its main descriptive fields using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is a single object; do not subscript.

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Date Published: {metadata.datePublished}")
print(f"License: {metadata.license}")

## 2. Data Overview
List available record sets, field `@id`s, and get an initial sense of the dataset's structure.

> **Note:** All record sets, fields, and columns are referenced by their `@id` according to the schema.

In [ ]:
# Explore available record sets in the dataset
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    print("Available Record Sets:")
    for rs in metadata.recordSet:
        print(f"- @id: {rs['@id'] if isinstance(rs, dict) and '@id' in rs else str(rs)}")
else:
    # Fallback: try to discover record sets via the mlcroissant API
    from mlcroissant._dataset import get_record_sets
    record_sets = get_record_sets(dataset.schema)
    print("Discovered Record Sets:")
    for rs in record_sets:
        print(f"- @id: {rs['@id']}")
        # List fields/columns
        if 'field' in rs:
            print("  Fields:")
            fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
            for field in fields:
                if isinstance(field, dict) and '@id' in field:
                    print(f"    - {field['@id']}")
                else:
                    print(f"    - {field}")
        # Optionally print more metadata here

## 3. Data Extraction
Load full data from a specific record set into a Pandas DataFrame for further analysis.

> All entities are referenced by their `@id` according to the schema.

In [ ]:
# Identify record set @ids
from mlcroissant._dataset import get_record_sets
record_sets = get_record_sets(dataset.schema)
record_set_ids = [rs['@id'] for rs in record_sets]
print("Record set @ids:", record_set_ids)

# Load dataframes for each record set by @id
dataframes = {}
for rs_id in record_set_ids:
    try:
        # Records yields dicts for each record; load into a DataFrame
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded {len(df)} records from record set '@id': {rs_id}")
        else:
            print(f"No records found for record set '@id': {rs_id}")
    except Exception as e:
        print(f"Failed to load records for '@id' {rs_id}: {e}")

# Show columns and preview the first few rows from the first available DataFrame
if dataframes:
    sample_rs_id = next(iter(dataframes))
    print(f"\nRecord set columns for '@id': {sample_rs_id}")
    print(dataframes[sample_rs_id].columns.tolist())
    dataframes[sample_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
We will filter, normalize, and group data from a numeric field within the loaded record set.

> Again, the field and record set are referenced by their schema `@id`.

*Modify the values of `numeric_field_id` and `group_field_id` if needed, based on the dataset structure discovered during the previous steps.*

In [ ]:
# Pick a loaded DataFrame and examine numeric fields
import numpy as np

if dataframes:
    # Use the same sample record set @id
    record_set_id = sample_rs_id
    df = dataframes[record_set_id]

    # Try to auto-select a numeric column by dtype
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"Numeric fields found in record set '@id': {record_set_id}: {numeric_cols}")
    numeric_field_id = numeric_cols[0] if numeric_cols else None

    if numeric_field_id is not None:
        threshold = df[numeric_field_id].mean() if not pd.isnull(df[numeric_field_id].mean()) else 10

        # Filter records with the numeric field above the threshold
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records where '{numeric_field_id}' > {threshold:.2f}:")
        print(filtered_df.head())

        # Create a normalized column
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to group by a likely group/categorical field (other than the numeric field)
        group_field_candidates = [col for col in df.columns if col != numeric_field_id and df[col].dtype == object]
        group_field_id = group_field_candidates[0] if group_field_candidates else None
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean '{numeric_field_id}' by '{group_field_id}':")
            print(grouped_df.head())
    else:
        print(f"No numeric fields found in record set '@id': {record_set_id}. Consider inspecting the DataFrame to find a suitable field.")

## 5. Visualization
Visualize distributions or relationships between numeric and group fields if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of '{numeric_field_id}' in '{record_set_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()
    
    if group_field_id:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"'{numeric_field_id}' by '{group_field_id}' in '{record_set_id}'")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No suitable numeric field to visualize.")

## 6. Conclusion
- We have demonstrated how to load and inspect a FAIR<sup>2</sup>-described dataset using `mlcroissant`, referencing all entities by their `@id` fields.
- We explored available record sets and fields, extracted records, performed basic EDA with filtering and normalization, and visualized selected fields.
- For further analysis, consult the dataset's Croissant schema and documentation to precisely identify domain-relevant entity `@id`s for more targeted processing.